<a href="https://colab.research.google.com/github/Raghukilli77/25P35A0577_DriveReady_OPC_2026/blob/main/gympilotaiagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install langchain_groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.4/591.4 kB 9.7 MB/s  0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.20
    Uninstalling langchain-protocol-0.0.20:
      Successfully uninstalled langchain-protocol-0.0.20
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain_groq]


In [2]:
from langchain_groq import ChatGroq

In [7]:
import os
from google.colab import userdata

try:
  os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except Exception:
    print("please add GROQ_API_KEY to your colab secrets")

#llm installing

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.9
)

In [5]:
system_prompt = """You are GymPilot — Intelligent Gym Management Assistant, built for gym owners using GymLane (gymlane.in).

ROLE:
You are a friendly, upbeat, and witty AI gym buddy and operational assistant dedicated to helping gym owners manage daily fitness center operations, member records, and business insights accurately—with a light touch of gym humor.

PURPOSE:
You exist to help gym owners quickly check member counts, track active and expired memberships, verify pricing plans, analyze membership statistics, and get actionable strategies to improve member retention—all while keeping the vibe positive and engaging.

KNOWLEDGE:
- You understand GymPilot's core gym management domain: Member IDs, member names, membership plans, monthly fees, membership statuses (Active, Expired), and expiry dates.
- You provide practical, data-grounded advice on gym operations, renewal reminders, and member retention.
- You recognize that real-time database/tool lookups are not connected yet in this foundation stage, so you rely strictly on the gym data and conversation context provided to you.

RULES:
1. Never invent or guess member information (IDs, names, statuses, or expiry dates).
2. Never invent or estimate financial figures, revenue totals, or plan prices.
3. Never expose or reference another gym's information; maintain strict single-gym data privacy.
4. Never claim an action (such as updating a record, sending a reminder, or processing a payment) was completed without explicit confirmation.
5. If requested information is missing or unavailable in the provided data or conversation history, admit it clearly and state what information is needed.
6. Keep jokes lighthearted, clean, and gym-related—never let humor get in the way of accurate data or mock specific members.

FORMAT:
- Keep responses friendly, clear, and scannable, weaving in a quick gym pun or lighthearted joke when natural (for example, in greetings or closing lines).
- Use bullet points or clean Markdown tables when listing multiple members, plans, or financial statistics.
- Lead directly with the answer so the owner gets their numbers fast before the punchline.
"""

In [6]:
gym_members = [
    {
        "member_id": "MEM-101",
        "name": "Sandeep Reddy Vanga",
        "membership_plan": "Quarterly",
        "monthly_fee": 1500,
        "membership_status": "Active",
        "expiry_date": "2027-01-15",
    },
    {
        "member_id": "MEM-102",
        "name": "Ram Gopal Varma (RGV)",
        "membership_plan": "Annual",
        "monthly_fee": 1200,
        "membership_status": "Active",
        "expiry_date": "2027-08-10",
    },
    {
        "member_id": "MEM-103",
        "name": "Brahmanandam",
        "membership_plan": "Monthly",
        "monthly_fee": 1800,
        "membership_status": "Expired",
        "expiry_date": "2026-11-10",
    },
    {
        "member_id": "MEM-104",
        "name": "SS Rajamouli",
        "membership_plan": "Personal Training",
        "monthly_fee": 4500,
        "membership_status": "Active",
        "expiry_date": "2026-12-20",
    },
    {
        "member_id": "MEM-105",
        "name": "Sunil",
        "membership_plan": "Monthly",
        "monthly_fee": 1800,
        "membership_status": "Expired",
        "expiry_date": "2026-11-22",
    },
]

# Quick check
for member in gym_members:
  print(member)

{'member_id': 'MEM-101', 'name': 'Sandeep Reddy Vanga', 'membership_plan': 'Quarterly', 'monthly_fee': 1500, 'membership_status': 'Active', 'expiry_date': '2027-01-15'}
{'member_id': 'MEM-102', 'name': 'Ram Gopal Varma (RGV)', 'membership_plan': 'Annual', 'monthly_fee': 1200, 'membership_status': 'Active', 'expiry_date': '2027-08-10'}
{'member_id': 'MEM-103', 'name': 'Brahmanandam', 'membership_plan': 'Monthly', 'monthly_fee': 1800, 'membership_status': 'Expired', 'expiry_date': '2026-11-10'}
{'member_id': 'MEM-104', 'name': 'SS Rajamouli', 'membership_plan': 'Personal Training', 'monthly_fee': 4500, 'membership_status': 'Active', 'expiry_date': '2026-12-20'}
{'member_id': 'MEM-105', 'name': 'Sunil', 'membership_plan': 'Monthly', 'monthly_fee': 1800, 'membership_status': 'Expired', 'expiry_date': '2026-11-22'}


In [10]:
import json
from langchain_core.messages import SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# 1. Convert your gym_members list into clean text so the AI can read it
gym_data_text = json.dumps(gym_members, indent=2)

# 2. Build the prompt with SystemMessage (avoids JSON { } variable errors) + history
prompt = ChatPromptTemplate.from_messages([
    SystemMessage(
        content=system_prompt + "\n\nCURRENT GYM MEMBER DATA:\n" + gym_data_text
    ),
    MessagesPlaceholder("history"),  # the chat so far goes here
    ("human", "{question}"),
])

chain = prompt | llm | StrOutputParser()

# 3. Conversation memory + ask() helper
history = []


def ask(question: str) -> str:
  answer = chain.invoke({"history": history, "question": question})
  history.extend([("human", question), ("ai", answer)])
  return answer

In [ ]:
print(ask("who are You"))

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2386745435.py, line 1)